# Supplementary Source Ingest

Add three new cultural-salience signals beyond Jeopardy, Wikipedia pageviews, and crossword frequency:

1. **Wiktextract proverbs & idioms** — canonical English sayings from Wiktionary
2. **Bartlett's Familiar Quotations** — enduring quotations from the public-domain 9th edition (1901)
3. **Wikidata SPARQL** — globally notable cultural figures and geographic places by sitelink count

Each section outputs a standardized CSV to `data/`.

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import json
import time
import gzip
import os
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../data"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CulturalLiteracyProject/1.0 (research; contact: abrunk@github)"})

def normalize(text):
    """Normalize a term for matching (reused from notebook 05)."""
    if not isinstance(text, str):
        return ""
    s = text.lower().strip()
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"[,;:!?'\"()\[\]]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("Ready.")

---
## Section 1: Wiktextract Proverbs & Idioms

Download English Wiktionary JSON lines from [kaikki.org](https://kaikki.org/dictionary/English/),
stream and filter for entries categorized as English proverbs or English idioms.

In [ ]:
# Download the English Wiktionary JSONL (gzipped) from kaikki.org
WIKTEXTRACT_URL = "https://kaikki.org/dictionary/English/kaikki.org-dictionary-English.jsonl.gz"
WIKTEXTRACT_CACHE = os.path.join(DATA_DIR, "kaikki_english.jsonl.gz")

if not os.path.exists(WIKTEXTRACT_CACHE):
    print(f"Downloading Wiktextract English dump (~280 MB)...")
    resp = SESSION.get(WIKTEXTRACT_URL, stream=True, timeout=300)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    downloaded = 0
    with open(WIKTEXTRACT_CACHE, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB", end="\r")
    print(f"\nSaved to {WIKTEXTRACT_CACHE}")
else:
    print(f"Using cached file: {WIKTEXTRACT_CACHE} ({os.path.getsize(WIKTEXTRACT_CACHE) / 1e6:.0f} MB)")

In [ ]:
# Stream and filter for proverbs and idioms
# Proverbs: identified by pos == "proverb"
# Idioms: multi-word phrases with "idiomatic" in sense-level tags

proverbs_idioms = []
n_scanned = 0

with gzip.open(WIKTEXTRACT_CACHE, "rt", encoding="utf-8") as f:
    for line in f:
        n_scanned += 1
        if n_scanned % 200_000 == 0:
            print(f"  Scanned {n_scanned:,} entries, found {len(proverbs_idioms):,} proverbs/idioms", end="\r")
        
        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            continue
        
        word = entry.get("word", "").strip()
        if not word or len(word) < 3:
            continue
        
        pos = entry.get("pos", "")
        senses = entry.get("senses", [])
        
        # Extract first sense definition
        first_gloss = ""
        if senses:
            glosses = senses[0].get("glosses", [])
            if glosses:
                first_gloss = glosses[0]
        
        # Method 1: pos == "proverb"
        if pos == "proverb":
            proverbs_idioms.append({
                "headword": word,
                "sub_type": "PROVERB",
                "gloss": first_gloss,
            })
            continue
        
        # Method 2: multi-word phrases tagged "idiomatic" in any sense
        if " " not in word:
            continue  # skip single-word entries for idioms
        
        is_idiomatic = False
        for s in senses:
            tags = s.get("tags", [])
            if "idiomatic" in tags:
                is_idiomatic = True
                # Use this sense's gloss if we don't have one yet
                if not first_gloss:
                    sg = s.get("glosses", [])
                    if sg:
                        first_gloss = sg[0]
                break
        
        if is_idiomatic:
            proverbs_idioms.append({
                "headword": word,
                "sub_type": "IDIOM",
                "gloss": first_gloss,
            })

print(f"\nScanned {n_scanned:,} total entries")
print(f"Found {len(proverbs_idioms):,} proverbs/idioms")

In [ ]:
pi_df = pd.DataFrame(proverbs_idioms)

# Normalize headwords
pi_df["norm"] = pi_df["headword"].apply(normalize)

# Deduplicate on normalized form (some entries appear in both proverb and idiom cats)
pi_df = pi_df.drop_duplicates(subset="norm", keep="first").reset_index(drop=True)

# Assign entity_type
pi_df["entity_type"] = "PROVERB_IDIOM"

# Cross-reference against existing crossword idiom clues
xw_idioms = pd.read_csv(os.path.join(DATA_DIR, "xd_crossword_idiom_clues.csv"))
# Build a set of normalized crossword idiom answers for matching
xw_idiom_answers = set(xw_idioms["answer"].str.lower().str.replace(" ", "").str.replace("_", ""))

def xw_key(text):
    if not isinstance(text, str):
        return ""
    return text.lower().replace(" ", "").replace("_", "").replace("-", "").replace(".", "").replace("'", "")

pi_df["in_crossword_idioms"] = pi_df["headword"].apply(xw_key).isin(xw_idiom_answers)

print(f"Deduplicated proverbs/idioms: {len(pi_df):,}")
print(f"  Proverbs: {(pi_df['sub_type'] == 'PROVERB').sum():,}")
print(f"  Idioms: {(pi_df['sub_type'] == 'IDIOM').sum():,}")
print(f"  Also in crossword idiom clues: {pi_df['in_crossword_idioms'].sum():,}")
print(f"\nSample proverbs:")
print(pi_df[pi_df["sub_type"] == "PROVERB"].head(10)[["headword", "gloss"]].to_string(index=False))
print(f"\nSample idioms:")
print(pi_df[pi_df["sub_type"] == "IDIOM"].head(10)[["headword", "gloss"]].to_string(index=False))

In [ ]:
# Export
pi_output = os.path.join(DATA_DIR, "wiktextract_proverbs_idioms.csv")
pi_df[["headword", "norm", "entity_type", "sub_type", "gloss", "in_crossword_idioms"]].to_csv(
    pi_output, index=False
)
print(f"Saved {len(pi_df):,} proverbs/idioms to {pi_output}")

---
## Section 2: Bartlett's Familiar Quotations

Download the 9th edition (1901, public domain) from Project Gutenberg #27889.
Parse author headers and extract the first line of each quotation.

In [ ]:
# Download Bartlett's from Project Gutenberg
BARTLETTS_URLS = [
    "https://www.gutenberg.org/cache/epub/27889/pg27889.txt",
    "https://www.gutenberg.org/files/27889/27889-0.txt",
]
BARTLETTS_CACHE = os.path.join(DATA_DIR, "bartletts_gutenberg.txt")

if not os.path.exists(BARTLETTS_CACHE):
    print("Downloading Bartlett's Familiar Quotations from Project Gutenberg...")
    for url in BARTLETTS_URLS:
        for attempt in range(3):
            try:
                print(f"  Trying {url} (attempt {attempt + 1})...")
                resp = SESSION.get(url, timeout=120)
                resp.raise_for_status()
                with open(BARTLETTS_CACHE, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                print(f"  Saved ({len(resp.text):,} chars)")
                break
            except Exception as e:
                print(f"  Failed: {e}")
                time.sleep(5)
        else:
            continue  # try next URL
        break  # success
    else:
        raise RuntimeError("Could not download Bartlett's from any URL")
else:
    print(f"Using cached file: {BARTLETTS_CACHE}")

with open(BARTLETTS_CACHE, "r", encoding="utf-8") as f:
    bartletts_text = f.read()
print(f"Loaded {len(bartletts_text):,} characters")

In [ ]:
# Parse Bartlett's: detect author headers and extract quotations
# The Gutenberg text has a structure where author names appear as headers
# followed by quotation blocks. The format varies but generally:
# - Author names appear on their own line, often in ALL CAPS or title case
# - Quotations are indented or follow specific patterns
# - Footnote markers and page references are interspersed

lines = bartletts_text.split("\n")

# Find the start/end of actual content (skip Gutenberg header/footer)
content_start = 0
content_end = len(lines)
for i, line in enumerate(lines):
    if "*** START OF" in line:
        content_start = i + 1
    if "*** END OF" in line:
        content_end = i
        break

content_lines = lines[content_start:content_end]
print(f"Content lines: {len(content_lines):,} (from line {content_start} to {content_end})")
print(f"\nFirst 30 content lines:")
for line in content_lines[:30]:
    print(f"  |{line}|")

In [ ]:
# Parse quotations from Bartlett's
# Strategy: identify author headers and quotation blocks
# Author headers are typically ALL CAPS lines with birth/death dates
# Quotations often start with indentation or after a blank line

# Pattern for author lines: ALL CAPS name possibly followed by dates
author_pattern = re.compile(
    r"^\s*([A-Z][A-Z .'-]{3,})\s*"
    r"(?:\[?\d{3,4}|\(?\d{3,4}|_\d{3,4}|\d{3,4}\s*[-–])",
)

# Also match simpler ALL CAPS lines that look like names (no dates)
allcaps_name_pattern = re.compile(r"^\s*([A-Z][A-Z .'-]{5,})\s*$")

# Pattern for footnote/page reference lines (to skip)
footnote_pattern = re.compile(r"^\s*(Footnote \d|\[\d|Page \d|_Page)")

quotations = []
current_author = "Unknown"
current_quote_lines = []
in_quote = False

def flush_quote():
    """Save accumulated quote lines as a quotation."""
    if not current_quote_lines:
        return
    text = " ".join(current_quote_lines).strip()
    # Clean up: remove footnote markers, excess whitespace
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) >= 10 and len(text) <= 500:
        # Take first line/sentence as the quotation key
        first_line = text.split(".")[0].strip() + "." if "." in text else text
        if len(first_line) > 200:
            first_line = first_line[:200]
        quotations.append({
            "quotation": first_line,
            "full_text": text[:500],
            "author": current_author,
        })

for line in content_lines:
    stripped = line.strip()
    
    # Skip empty lines and footnotes
    if not stripped:
        if in_quote:
            flush_quote()
            current_quote_lines = []
            in_quote = False
        continue
    
    if footnote_pattern.match(stripped):
        continue
    
    # Check for author header
    author_match = author_pattern.match(stripped)
    if not author_match:
        # Try all-caps name pattern
        allcaps_match = allcaps_name_pattern.match(stripped)
        if allcaps_match and len(stripped) < 60:
            author_match = allcaps_match
    
    if author_match:
        # Flush any pending quote
        if in_quote:
            flush_quote()
            current_quote_lines = []
            in_quote = False
        # Extract author name
        name = author_match.group(1).strip()
        # Title-case it
        current_author = name.title().strip(". ")
        continue
    
    # Otherwise, this is part of a quotation
    # Skip lines that are just numbers or very short
    if re.match(r"^\d+\.?$", stripped):
        continue
    
    if not in_quote:
        in_quote = True
        current_quote_lines = []
    current_quote_lines.append(stripped)

# Flush last quote
flush_quote()

print(f"Extracted {len(quotations):,} quotations")
print(f"From {len(set(q['author'] for q in quotations)):,} authors")
print(f"\nSample quotations:")
for q in quotations[:10]:
    print(f"  [{q['author']}] {q['quotation'][:80]}")

In [ ]:
bq_df = pd.DataFrame(quotations)

# Normalize quotation text for matching
bq_df["norm"] = bq_df["quotation"].apply(normalize)

# Deduplicate on normalized quotation
bq_df = bq_df.drop_duplicates(subset="norm", keep="first").reset_index(drop=True)

# Assign entity type
bq_df["entity_type"] = "QUOTATION"

# Normalize author names for linking to existing Jeopardy PERSON concepts
bq_df["author_norm"] = bq_df["author"].apply(normalize)

# Load existing Jeopardy concepts to link authors
jeopardy = pd.read_csv(os.path.join(DATA_DIR, "jeopardy_concepts_v3.csv"), index_col=0,
                        engine="python", on_bad_lines="skip")
jeopardy_persons = jeopardy[jeopardy["entity_type"] == "PERSON"]
person_norms = set(jeopardy_persons["concept"].apply(normalize))
person_display_norms = set(jeopardy_persons["display_name"].apply(normalize))
all_person_norms = person_norms | person_display_norms

bq_df["author_in_jeopardy"] = bq_df["author_norm"].isin(all_person_norms)

# Count quotations per author
author_counts = bq_df["author"].value_counts()

print(f"Deduplicated quotations: {len(bq_df):,}")
print(f"Unique authors: {bq_df['author'].nunique():,}")
print(f"Authors also in Jeopardy PERSON concepts: {bq_df['author_in_jeopardy'].sum():,} quotations")
print(f"\nTop 20 authors by quotation count:")
for author, count in author_counts.head(20).items():
    in_j = "[J]" if normalize(author) in all_person_norms else "   "
    print(f"  {in_j} {author:30s} {count:>4}")

In [ ]:
# Export
bq_output = os.path.join(DATA_DIR, "bartletts_quotations.csv")
bq_df[["quotation", "norm", "full_text", "author", "author_norm", "entity_type",
       "author_in_jeopardy"]].to_csv(bq_output, index=False)
print(f"Saved {len(bq_df):,} quotations to {bq_output}")

---
## Section 3: Wikidata SPARQL — Notable Figures & Places

Query Wikidata for globally notable cultural figures and geographic places,
using sitelink count as a proxy for cross-cultural recognition.

Two queries:
1. **Cultural figures**: writers, philosophers, scientists, political leaders, etc. with sitelinks >= 40
2. **Notable places**: cities (pop >= 100k or capital), rivers, mountains, lakes with sitelinks >= 20

In [ ]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

def run_sparql(query, retries=3, delay=30):
    """Run a SPARQL query against Wikidata and return results as a DataFrame."""
    for attempt in range(retries):
        try:
            resp = SESSION.get(
                WIKIDATA_SPARQL,
                params={"query": query, "format": "json"},
                timeout=120,
            )
            resp.raise_for_status()
            data = resp.json()
            results = data["results"]["bindings"]
            rows = []
            for r in results:
                row = {}
                for key, val in r.items():
                    row[key] = val.get("value", "")
                rows.append(row)
            return pd.DataFrame(rows)
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                print(f"  Retrying in {delay}s...")
                time.sleep(delay)
    return pd.DataFrame()

print("SPARQL helper ready.")

In [ ]:
# Query 1: Notable cultural figures
# The full query with 30 occupations times out on Wikidata's public endpoint.
# Split into smaller batches of occupations.

OCCUPATION_BATCHES = [
    # Batch 1: literary/intellectual
    ["wd:Q36180", "wd:Q4964182", "wd:Q49757", "wd:Q214917", "wd:Q1781198",
     "wd:Q11513337", "wd:Q1234713"],
    # Batch 2: science
    ["wd:Q901", "wd:Q593644", "wd:Q169470", "wd:Q170790", "wd:Q188094",
     "wd:Q1622272"],
    # Batch 3: political/military
    ["wd:Q82955", "wd:Q116", "wd:Q47064", "wd:Q189290", "wd:Q40348"],
    # Batch 4: arts/entertainment
    ["wd:Q483501", "wd:Q36834", "wd:Q2526255", "wd:Q33999", "wd:Q1028181",
     "wd:Q11774202", "wd:Q177220"],
    # Batch 5: other
    ["wd:Q1930187", "wd:Q131524", "wd:Q43845", "wd:Q15981151", "wd:Q11900058"],
]

all_figures = []
for i, batch in enumerate(OCCUPATION_BATCHES):
    occ_values = " ".join(batch)
    query = f"""
SELECT ?item ?itemLabel ?itemDescription ?sitelinks ?occupationLabel WHERE {{
  ?item wdt:P31 wd:Q5 .
  ?item wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= 40)
  ?item wdt:P106 ?occupation .
  VALUES ?occupation {{ {occ_values} }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
}}
ORDER BY DESC(?sitelinks)
"""
    print(f"  Batch {i+1}/{len(OCCUPATION_BATCHES)}: {len(batch)} occupations...", end=" ")
    result = run_sparql(query)
    print(f"{len(result):,} rows")
    if len(result) > 0:
        all_figures.append(result)
    time.sleep(10)  # rate limit

if all_figures:
    figures_raw = pd.concat(all_figures, ignore_index=True)
    print(f"\nTotal raw figure results: {len(figures_raw):,}")
else:
    figures_raw = pd.DataFrame()
    print("\nNo figures returned from any batch.")

In [ ]:
# Process figures: deduplicate by QID (multiple occupations create multiple rows)
# Load existing QIDs upfront (used by both figures and places sections)
existing_linking = pd.read_csv(os.path.join(DATA_DIR, "wikidata_linking.csv"))
existing_qids = set(existing_linking["qid"].dropna())
print(f"Existing Wikidata QIDs from Jeopardy linking: {len(existing_qids):,}")

if len(figures_raw) > 0:
    # Extract QID from item URI
    figures_raw["qid"] = figures_raw["item"].str.extract(r"(Q\d+)$")
    figures_raw["sitelinks"] = pd.to_numeric(figures_raw["sitelinks"], errors="coerce")
    
    # Group by QID: take max sitelinks, collect occupations
    agg_dict = {
        "label": ("itemLabel", "first"),
        "sitelinks": ("sitelinks", "max"),
        "occupations": ("occupationLabel", lambda x: "; ".join(sorted(set(x)))),
    }
    # Add optional columns if present
    if "itemDescription" in figures_raw.columns:
        agg_dict["description"] = ("itemDescription", "first")
    
    figures = figures_raw.groupby("qid").agg(**agg_dict).reset_index()
    
    if "description" not in figures.columns:
        figures["description"] = ""
    
    figures["norm"] = figures["label"].apply(normalize)
    figures["entity_type"] = "PERSON"
    figures = figures.sort_values("sitelinks", ascending=False).reset_index(drop=True)
    
    figures["already_in_jeopardy"] = figures["qid"].isin(existing_qids)
    
    # Add placeholder columns for birth/death/country (not in batched query)
    for col in ["birth_year", "death_year", "country"]:
        if col not in figures.columns:
            figures[col] = ""
    
    print(f"Unique figures: {len(figures):,}")
    print(f"Already in Jeopardy (via QID match): {figures['already_in_jeopardy'].sum():,}")
    print(f"New figures: {(~figures['already_in_jeopardy']).sum():,}")
    print(f"\nTop 30 by sitelinks:")
    print(figures.head(30)[["qid", "label", "sitelinks", "occupations",
                            "already_in_jeopardy"]].to_string(index=False))
else:
    print("No figures returned — check query.")
    figures = pd.DataFrame()

In [ ]:
# Export figures
if len(figures) > 0:
    fig_output = os.path.join(DATA_DIR, "wikidata_global_figures.csv")
    figures[["qid", "label", "norm", "description", "entity_type", "sitelinks",
             "occupations", "birth_year", "death_year", "country",
             "already_in_jeopardy"]].to_csv(fig_output, index=False)
    print(f"Saved {len(figures):,} figures to {fig_output}")

In [ ]:
# Query 2: Notable places — cities, rivers, mountains, lakes
# Split into sub-queries to avoid Wikidata timeout

# 2a: Cities — population >= 100k or national capital, sitelinks >= 20
# Use direct P31 to avoid subclass traversal timeout
CITIES_QUERY = """
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?sitelinks ?population ?countryLabel WHERE {
  {
    ?item wdt:P31 wd:Q515 .    # instance of city
    ?item wdt:P1082 ?population .
    FILTER(?population >= 100000)
  } UNION {
    ?item wdt:P31 wd:Q515 .    # instance of city
    ?item wdt:P1376 ?adminTerritory .     # capital of some territory
  }
  ?item wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= 20)
  OPTIONAL { ?item wdt:P17 ?country . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
ORDER BY DESC(?sitelinks)
"""

print("Querying Wikidata for notable cities...")
cities_raw = run_sparql(CITIES_QUERY)
print(f"Cities raw: {len(cities_raw):,} rows")

In [ ]:
# 2b: Geographic features — rivers, mountains, lakes with sitelinks >= 20
# Use direct P31 (instance of) to avoid timeout from subclass traversal
GEO_FEATURES_QUERY = """
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?sitelinks ?featureTypeLabel WHERE {
  VALUES ?featureType {
    wd:Q4022     # river
    wd:Q8502     # mountain
    wd:Q23397    # lake
    wd:Q5107     # continent
    wd:Q23442    # island
    wd:Q8072     # volcano
    wd:Q34763    # peninsula
    wd:Q185113   # plateau
    wd:Q35666    # archipelago
    wd:Q165      # sea
    wd:Q9430     # ocean
    wd:Q39816    # valley
    wd:Q35509    # cave
    wd:Q180874   # glacier
  }
  ?item wdt:P31 ?featureType .
  ?item wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= 20)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
ORDER BY DESC(?sitelinks)
"""

print("Querying Wikidata for notable geographic features...")
geo_raw = run_sparql(GEO_FEATURES_QUERY)
print(f"Geographic features raw: {len(geo_raw):,} rows")

In [ ]:
# Combine and process places
places_dfs = []

if len(cities_raw) > 0:
    cities_raw["qid"] = cities_raw["item"].str.extract(r"(Q\d+)$")
    cities_raw["sitelinks"] = pd.to_numeric(cities_raw["sitelinks"], errors="coerce")
    cities_raw["population"] = pd.to_numeric(cities_raw.get("population", 0), errors="coerce")
    
    cities = cities_raw.groupby("qid").agg(
        label=("itemLabel", "first"),
        description=("itemDescription", "first"),
        sitelinks=("sitelinks", "max"),
        population=("population", "max"),
        country=("countryLabel", "first"),
    ).reset_index()
    cities["feature_type"] = "city"
    places_dfs.append(cities)
    print(f"Unique cities: {len(cities):,}")

if len(geo_raw) > 0:
    geo_raw["qid"] = geo_raw["item"].str.extract(r"(Q\d+)$")
    geo_raw["sitelinks"] = pd.to_numeric(geo_raw["sitelinks"], errors="coerce")
    
    geo_features = geo_raw.groupby("qid").agg(
        label=("itemLabel", "first"),
        description=("itemDescription", "first"),
        sitelinks=("sitelinks", "max"),
        feature_type=("featureTypeLabel", "first"),
    ).reset_index()
    geo_features["population"] = np.nan
    geo_features["country"] = ""
    places_dfs.append(geo_features)
    print(f"Unique geo features: {len(geo_features):,}")

if places_dfs:
    places = pd.concat(places_dfs, ignore_index=True)
    # Deduplicate by QID (a city near a volcano might appear in both)
    places = places.sort_values("sitelinks", ascending=False).drop_duplicates(subset="qid", keep="first")
    places["norm"] = places["label"].apply(normalize)
    places["entity_type"] = places["feature_type"].map(
        lambda x: "GPE" if x == "city" else "LOC"
    )
    places = places.reset_index(drop=True)
    
    # Deduplicate against existing wikidata_linking.csv
    places["already_in_jeopardy"] = places["qid"].isin(existing_qids)
    
    print(f"\nTotal unique places: {len(places):,}")
    print(f"Already in Jeopardy (via QID): {places['already_in_jeopardy'].sum():,}")
    print(f"New places: {(~places['already_in_jeopardy']).sum():,}")
    print(f"\nBy feature type:")
    print(places["feature_type"].value_counts().to_string())
    print(f"\nTop 30 by sitelinks:")
    print(places.head(30)[["qid", "label", "sitelinks", "feature_type", "country",
                           "already_in_jeopardy"]].to_string(index=False))
else:
    print("No places returned — check queries.")
    places = pd.DataFrame()

In [ ]:
# Export places
if len(places) > 0:
    places_output = os.path.join(DATA_DIR, "wikidata_notable_places.csv")
    places[["qid", "label", "norm", "description", "entity_type", "feature_type",
            "sitelinks", "population", "country",
            "already_in_jeopardy"]].to_csv(places_output, index=False)
    print(f"Saved {len(places):,} places to {places_output}")

---
## Summary

In [ ]:
print("=" * 65)
print("SUPPLEMENTARY INGEST SUMMARY")
print("=" * 65)
print(f"\n1. Wiktextract Proverbs & Idioms: {len(pi_df):,}")
print(f"   - Proverbs: {(pi_df['sub_type'] == 'PROVERB').sum():,}")
print(f"   - Idioms: {(pi_df['sub_type'] == 'IDIOM').sum():,}")
print(f"   - Output: data/wiktextract_proverbs_idioms.csv")

print(f"\n2. Bartlett's Familiar Quotations: {len(bq_df):,}")
print(f"   - Unique authors: {bq_df['author'].nunique():,}")
print(f"   - Authors also in Jeopardy: {bq_df[bq_df['author_in_jeopardy']]['author'].nunique():,}")
print(f"   - Output: data/bartletts_quotations.csv")

if len(figures) > 0:
    print(f"\n3a. Wikidata Cultural Figures: {len(figures):,}")
    print(f"   - Already in Jeopardy: {figures['already_in_jeopardy'].sum():,}")
    print(f"   - New: {(~figures['already_in_jeopardy']).sum():,}")
    print(f"   - Output: data/wikidata_global_figures.csv")

if len(places) > 0:
    print(f"\n3b. Wikidata Notable Places: {len(places):,}")
    print(f"   - Already in Jeopardy: {places['already_in_jeopardy'].sum():,}")
    print(f"   - New: {(~places['already_in_jeopardy']).sum():,}")
    print(f"   - Output: data/wikidata_notable_places.csv")

print(f"\nAll outputs saved to {DATA_DIR}/")